# Second layer: context adjustment on top of the threes model

**What this is.** The MNLogit PMF stays fixed as the base. A small second model tilts it for tonight's context and widens it where it is too narrow:

$$\log p'(k) = \log p_{model}(k) + k\,(x\beta) + k^2\,(z\gamma) + \delta\,1\{k=0\} - \log Z$$

- $k\,(x\beta)$ moves the mean (lineup context).
- $k^2\,(z\gamma)$ changes the spread; $\delta$ adds or removes zero mass. These are the dispersion fix.
- Priors are Normal(0, $\sigma$) on every coefficient, so the base model stands unless the data says otherwise. The fit is the posterior mode (MAP), which is a ridge-penalized fit.

**Order of work** (each section ends with a *Work through* list):
1. Data: base PMFs + outcomes
2. How the base scores
3. The tilt, and what the priors imply (prior predictive)
4. Step A: dispersion only
5. Features: who is out, vacated usage / creation / attempts, Gini, per-36 deviation; where the zero miss sits
6. Step B: lineup terms on top of dispersion
7. Coefficient uncertainty (quadratic approximation) and prior sensitivity
8. Calibration at the betting lines
9. Save base and adjusted side by side

**Reading alongside (Statistical Rethinking, 2nd ed.):** ch. 4 for the quadratic approximation (section 7), ch. 7 for regularizing priors and out-of-sample log score (sections 3, 4, 6), ch. 11-12 for counts, over-dispersion and zero inflation (sections 2, 4), ch. 13 if you later add player-level effects.

**Limits to keep in mind**
- Stored predictions come from two different models (the 2025-26 model through April, the retrained model from Oct 20). `FIT_SEASONS` controls which are used.
- The sample is small. Treat every result here as a first look until the walk-forward re-score exists.
- `predictions` holds the day's last run, so it is the 11:30 PMF, not always the one bet on.

In [ ]:
import os, sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

pd.set_option('display.width', 200)
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'Notebooks' else os.getcwd()
DB = os.environ.get('NBA_DB_PATH', os.path.join(ROOT, 'nba', 'data', 'database', 'nba.db'))
conn = sqlite3.connect('file:{}?mode=ro'.format(DB), uri=True)   # read-only: nothing here writes to the database

K = np.arange(10)                 # outcomes 0..9 (9 = cap)
FIT_SEASONS = ['2026-27']         # add '2025-26' to include the March-April predictions from the old model
TRAIN_FRAC = 0.7                  # first 70% of dates train, the rest test
HIST_START = '2025-10-01'         # box-score history used to build features
ROT_MIN, ROT_WINDOW, TOP_N = 20, 10, 8   # rotation = 20+ min per game over the team's last 10 games; Gini over the top 8
CLUSTER = 'player_id'             # SE clustering for paired scores ('game_id' to compare)
print(DB)

## 1. Data: base PMFs and outcomes
One row per player-game: the stored PMF (`p0..p9`), the actual threes made (`y`, capped at 9), and the game keys. DNPs drop out in the join (books void them).

In [ ]:
pcols = [r[1] for r in conn.execute('PRAGMA table_info(predictions)')]
dcol = 'game_date' if 'game_date' in pcols else 'date'      # dev has the game_date rename, prod may not yet

# predictions.player_id is INTEGER, plyrLogs.player_id is TEXT: cast here until the source types match
pred = pd.read_sql("""SELECT CAST(player_id AS TEXT) AS player_id, {} AS game_date, CAST(number AS INT) AS k, model_prob
                      FROM predictions WHERE market = 'threes'""".format(dcol), conn)
pmf = pred.pivot(index=['player_id', 'game_date'], columns='k', values='model_prob').dropna()
pmf.columns = ['p{}'.format(k) for k in pmf.columns]
print('PMF rows: {:,}   row sums: {:.4f} to {:.4f}'.format(len(pmf), pmf.sum(axis=1).min(), pmf.sum(axis=1).max()))

logs = pd.read_sql("""SELECT p.player_id, p.team_id, p.game_id, p.game_date, t.season, t.home, p.min, p.usagePercentage AS usg,
                             COALESCE(p.lc_fgm, 0) + COALESCE(p.rc_fgm, 0) + COALESCE(p.abv_fgm, 0) AS fg3m,
                             COALESCE(p.lc_fga, 0) + COALESCE(p.rc_fga, 0) + COALESCE(p.abv_fga, 0) AS fg3a
                      FROM plyrLogs p JOIN teamLog t USING (game_id, team_id)
                      WHERE p.game_date >= '{}' AND p.min > 0""".format(HIST_START), conn)
trk = pd.read_sql("""SELECT player_id, game_date, potential_assists AS pot_ast, catchshoot_fg3a AS cs3a, pullup_fg3a AS pu3a
                     FROM playerTracking WHERE game_date >= '{}'""".format(HIST_START), conn)

d = pmf.reset_index().merge(logs, on=['player_id', 'game_date'])
d = d[d.season.isin(FIT_SEASONS)].sort_values(['game_date', 'game_id', 'player_id']).reset_index(drop=True)
d['y'] = d.fg3m.clip(upper=9).astype(int)
PC = ['p{}'.format(k) for k in K]
d[PC] = d[PC].div(d[PC].sum(axis=1), axis=0)
print('player-games with outcomes: {:,}   dates: {} to {} ({} days)'.format(len(d), d.game_date.min(), d.game_date.max(), d.game_date.nunique()))
print('tracking rows by month:'); print(trk.groupby(trk.game_date.str[:7]).size().tail(4).to_string())

**Work through**
- Does the row count match what you expect (roughly players predicted per day x days)? If it is far short, check how many PMFs were dropped for a null class.
- Is tracking loaded for the dates you are fitting? If the last months are missing, `vac_crea` and `cs_share` below will be zero / empty.

## 2. How the base scores
Scores used throughout:
- **Log loss**: $-\log p(y)$. The quantity the fit minimizes; punishes a too-narrow PMF hardest.
- **RPS** (ranked probability score): squared distance between the predicted and observed CDFs. Respects the ordering of counts, so being off by one costs less than being off by four.

Differences between two models are paired per player-game, with standard errors clustered by `CLUSTER`.

In [ ]:
def scores(P, y):
    '''Per-row log loss and RPS. Input: P (n x 10 PMFs), y (n outcomes). Output: (logloss, rps) arrays.'''
    ll = -np.log(np.clip(P[np.arange(len(y)), y], 1e-12, None))
    rps = ((P.cumsum(axis=1) - (K[None, :] >= y[:, None])) ** 2).sum(axis=1)
    return ll, rps

def paired(a, b, cluster):
    '''Mean of a - b with a cluster-robust SE. Output: (mean, se).'''
    df = pd.DataFrame({'d': np.asarray(a) - np.asarray(b), 'c': np.asarray(cluster)})
    m = df.d.mean()
    g = df.groupby('c').d.agg(['sum', 'size'])
    G = len(g)
    return m, np.sqrt(G / (G - 1) * ((g['sum'] - g['size'] * m) ** 2).sum()) / len(df)

def describe(P, y, label):
    '''One-row summary of a set of PMFs against outcomes.'''
    mean = P @ K
    var = P @ K ** 2 - mean ** 2
    ll, rps = scores(P, y)
    return pd.Series({'n': len(y), 'mean pred': mean.mean(), 'mean actual': y.mean(), 'P(0) pred': P[:, 0].mean(), 'P(0) actual': (y == 0).mean(),
                      'model var': var.mean(), 'realized sq err': ((y - mean) ** 2).mean(), 'logloss': ll.mean(), 'rps': rps.mean()}, name=label)

P0, y = d[PC].values, d.y.values
describe(P0, y, 'base').to_frame().T.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(K - 0.2, P0.mean(axis=0), width=0.4, label='base PMF (average)')
ax.bar(K + 0.2, np.bincount(y, minlength=10) / len(y), width=0.4, label='actual')
ax.set_xticks(K); ax.set_xlabel('threes made'); ax.set_ylabel('share of player-games'); ax.legend(frameon=False)
plt.show()

**Work through**
- `model var` vs `realized sq err`: if the model were right, these match. A realized value above the model's means the PMF is too narrow (the Nov 8 finding was 1.46 vs 1.70).
- In the chart, where is the base short: only at 0, or at 0 and in the right tail? That tells you whether $\delta$ (zero mass) or $\gamma$ (spread) should carry the fix.

## 3. The tilt, and what the priors imply
`tilt` applies the formula at the top. Before fitting, look at what the priors allow: draw coefficients from the priors, apply them to the average PMF, and see how far P(0) and the mean can move. If the prior lets P(0) run from 5% to 90%, it is not saying "the base model stands". Tighten or loosen `PRIOR` until the range is what you actually believe before seeing data.

In [ ]:
def tilt(P, eta, gam, delta):
    '''Tilted PMFs. Input: P (n x 10), eta and gam (length n or scalar: coefficients on k and k^2), delta (zero-mass shift). Output: n x 10.'''
    n = len(P)
    logq = np.log(np.clip(P, 1e-12, None)) + np.outer(np.broadcast_to(eta, n), K) + np.outer(np.broadcast_to(gam, n), K ** 2)
    logq[:, 0] += delta
    q = np.exp(logq - logq.max(axis=1, keepdims=True))
    return q / q.sum(axis=1, keepdims=True)

PRIOR = {'beta0': 0.10, 'beta': 0.15, 'gamma': 0.015, 'delta': 0.30}    # prior SDs. beta is per 1 SD of a standardized feature

rng = np.random.default_rng(7)
avg = P0.mean(axis=0, keepdims=True)
draws = np.vstack([tilt(avg, rng.normal(0, PRIOR['beta']), rng.normal(0, PRIOR['gamma']), rng.normal(0, PRIOR['delta'])) for _ in range(2000)])
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(draws[:, 0], bins=40); ax[0].axvline(avg[0, 0], color='k'); ax[0].set_title('prior predictive: P(0)')
ax[1].hist(draws @ K, bins=40); ax[1].axvline((avg @ K)[0], color='k'); ax[1].set_title('prior predictive: mean threes')
plt.show()
print('P(0) 5-95%: {:.2f} to {:.2f}    mean 5-95%: {:.2f} to {:.2f}'.format(*np.quantile(draws[:, 0], [.05, .95]), *np.quantile(draws @ K, [.05, .95])))

**Work through**
- Black line = the base. Is the 5-95% range a believable "how wrong could the base be for one player-game"? Adjust `PRIOR` and rerun.
- Set three of the four SDs to ~0 in turn to see what each coefficient does on its own. Note that $\gamma > 0$ fattens the right tail and raises the mean, which is why the fit also gets an intercept ($\beta_0$) to pull the mean back.

## 4. Step A: dispersion only
Three coefficients, no features: $\beta_0$ (re-centers the mean), $\gamma_0$ (spread), $\delta$ (zero mass). Fit on the training dates, score on the held-out dates.

The model is an exponential family with the base PMF as the reference measure, so the penalized log likelihood is concave and the gradient is just (expected - observed) for $k$, $k^2$ and $1\{k=0\}$.

In [ ]:
def unpack(theta, X, Z):
    pX, pZ = X.shape[1], Z.shape[1]
    return theta[:pX], theta[pX:pX + pZ], theta[-1]

def neg_log_post(theta, P, y, X, Z, sd):
    '''Negative log posterior and gradient. X: mean-tilt design (n x pX), Z: spread design (n x pZ), sd: prior SD per coefficient.'''
    b, g, dl = unpack(theta, X, Z)
    Q = tilt(P, X @ b, Z @ g, dl)
    val = -np.log(np.clip(Q[np.arange(len(y)), y], 1e-300, None)).sum() + 0.5 * ((theta / sd) ** 2).sum()
    grad = np.concatenate([X.T @ (Q @ K - y), Z.T @ (Q @ K ** 2 - y ** 2), [(Q[:, 0] - (y == 0)).sum()]]) + theta / sd ** 2
    return val, grad

def fit_layer(P, y, X, Z, sd):
    '''MAP fit. Output: theta = [beta..., gamma..., delta].'''
    res = minimize(neg_log_post, np.zeros(X.shape[1] + Z.shape[1] + 1), args=(P, y, X, Z, sd), jac=True, method='L-BFGS-B')
    if not res.success: print('optimizer:', res.message)
    return res.x

def predict_layer(theta, P, X, Z):
    b, g, dl = unpack(theta, X, Z)
    return tilt(P, X @ b, Z @ g, dl)

def design(df, mean_cols, disp_cols):
    '''Design matrices with intercepts first, and the matching prior-SD vector.'''
    one = np.ones((len(df), 1))
    X = np.hstack([one, df[mean_cols].values.astype(float)]) if mean_cols else one
    Z = np.hstack([one, df[disp_cols].values.astype(float)]) if disp_cols else one
    sd = np.array([PRIOR['beta0']] + [PRIOR['beta']] * len(mean_cols) + [PRIOR['gamma']] * (1 + len(disp_cols)) + [PRIOR['delta']])
    return X, Z, sd

dates = np.sort(d.game_date.unique())
CUTOFF = dates[int(len(dates) * TRAIN_FRAC)]
tr, te = (d.game_date < CUTOFF).values, (d.game_date >= CUTOFF).values
print('train: {:,} player-games before {}    test: {:,} from {}'.format(tr.sum(), CUTOFF, te.sum(), CUTOFF))

X, Z, sd = design(d, [], [])
thA = fit_layer(P0[tr], y[tr], X[tr], Z[tr], sd)
PA = predict_layer(thA, P0, X, Z)
print('beta0 {:+.3f}   gamma0 {:+.4f}   delta {:+.3f}'.format(*thA))
pd.concat([describe(P0[te], y[te], 'base (test)'), describe(PA[te], y[te], 'dispersion (test)')], axis=1).T.round(4)

In [ ]:
def compare(Pnew, Pref, mask, label):
    '''Paired score difference new - ref on the masked rows (negative = new is better), x1000.'''
    ll_n, rps_n = scores(Pnew[mask], y[mask]); ll_r, rps_r = scores(Pref[mask], y[mask])
    (ml, sl), (mr, sr) = paired(ll_n, ll_r, d.loc[mask, CLUSTER]), paired(rps_n, rps_r, d.loc[mask, CLUSTER])
    return pd.Series({'n': mask.sum(), 'logloss diff x1000': 1000 * ml, 'se': 1000 * sl, 'z': ml / sl, 'rps diff x1000': 1000 * mr, 'rps se': 1000 * sr, 'rps z': mr / sr}, name=label)

compare(PA, P0, te, 'dispersion vs base (test)').to_frame().T.round(2)

**Work through**
- Which of $\gamma_0$ and $\delta$ is doing the work? Refit with `PRIOR['gamma']` near 0, then with `PRIOR['delta']` near 0, and compare the held-out scores.
- Did the held-out P(0) and variance close the gap, or only part of it?
- The z here is the bar every later model has to clear **relative to this one**, not relative to the base. A lineup model that only beats the base may just be re-discovering the dispersion fix.
- Compare `CLUSTER = 'player_id'` with `'game_id'`. If the SE barely changes, the dependence is not where you clustered.

## 5. Features: tonight's context
All built from games **before** tonight (`tg_n < n`), per team.

**Who is out.** For each team-game, the candidates are everyone who played for the team in its previous `ROT_WINDOW` games. A *rotation* player averaged `ROT_MIN`+ minutes in the games he played.
- `out` = rotation player with no minutes tonight. This uses tonight's box score, so it is **diagnostic only**.
- `known_out` = `out` and he also missed the team's previous game. This is the stand-in for "known before the bet" until the injury-report pull exists. First-game absences are excluded on purpose: that is where the box score leaks.

**Team-level, from the `known_out` players**
- `vac_usg`: sum of (minutes per game / 48) x usage. One 36-minute, 30%-usage player out = 0.225.
- `vac_crea`: sum of potential assists per game.
- `vac_3pa`: $-\log(1 - \text{share of the team's 3PA those players took})$. This is the log bump every remaining player gets if attempts are redistributed in proportion.
- `gini`: concentration of expected 3PA across the top `TOP_N` active players by minutes.

**Player-level**
- `p36_dev`: log of recent (last 5) 3PA per minute minus log of season-to-date, each as (3PA + a) / (min + b) with a/b at the league rate. A deviation, because the base already has the level.
- `cs_share`: catch-and-shoot share of his tracked 3PA, last 20 games.

One thing the arithmetic shows: under proportional redistribution the share ratio is the **same for every active teammate** (that is `vac_3pa`). Who gains more than proportionally has to come from player traits interacting with it, e.g. `vac_crea x cs_share` (a catch-and-shoot player loses looks when the creator sits). That interaction is built below but left out of the default model for you to test.

In [ ]:
tg = logs[['team_id', 'season', 'game_id', 'game_date']].drop_duplicates().sort_values(['team_id', 'game_date'])
tg['tg_n'] = tg.groupby(['team_id', 'season']).cumcount()                      # team game number within the season
app = logs.merge(tg[['team_id', 'game_id', 'tg_n']], on=['team_id', 'game_id']).merge(trk, how='left', on=['player_id', 'game_date'])

# every prior appearance in the window, for every team-game
tonight = tg.rename(columns={'tg_n': 'n', 'game_id': 'gid'})[['team_id', 'season', 'n', 'gid']]
prior = app[['player_id', 'team_id', 'season', 'tg_n', 'min', 'fg3a', 'usg', 'pot_ast']].merge(tonight, on=['team_id', 'season'])
prior = prior[(prior.tg_n < prior.n) & (prior.tg_n >= prior.n - ROT_WINDOW)]
prior['usg_min'] = prior.usg * prior['min']
ros = prior.groupby(['gid', 'team_id', 'n', 'player_id']).agg(gp=('min', 'size'), min_sum=('min', 'sum'), fg3a_sum=('fg3a', 'sum'), usg_min=('usg_min', 'sum'),
                                                              pa_sum=('pot_ast', 'sum'), pa_n=('pot_ast', 'count'), last_n=('tg_n', 'max')).reset_index()
ros = ros.merge(app[['game_id', 'team_id', 'player_id']].rename(columns={'game_id': 'gid'}).assign(played=1), how='left', on=['gid', 'team_id', 'player_id'])
ros['played'] = ros.played.fillna(0)
ros['min_pg'] = ros.min_sum / ros.gp
ros['exp3a'] = ros.fg3a_sum / ros.gp                       # 3PA per game played
ros['usg_w'] = ros.usg_min / ros.min_sum                   # minutes-weighted usage
ros['pa_pg'] = (ros.pa_sum / ros.pa_n).fillna(0)           # potential assists per tracked game (0 where tracking is missing)
ros['rot'] = (ros.min_pg >= ROT_MIN) & (ros.gp >= 3)
ros['out'] = ros.rot & (ros.played == 0)
ros['known_out'] = ros.out & (ros.last_n < ros.n - 1)

def team_feats(r, flag, sfx=''):
    '''Team-game totals over the players flagged out. Output indexed by (gid, team_id).'''
    key = ['gid', 'team_id']
    o = r[r[flag]].assign(vu=lambda x: x.min_pg / 48 * x.usg_w)
    f = o.groupby(key).agg(n_out=('player_id', 'size'), vac_usg=('vu', 'sum'), vac_crea=('pa_pg', 'sum'), out3a=('exp3a', 'sum'))
    tot = r.assign(pg=r.fg3a_sum / r.n.clip(upper=ROT_WINDOW)).groupby(key).pg.sum().rename('tot3a')   # team 3PA per game over the window
    f = tot.to_frame().join(f).fillna(0)
    f['vac_3pa'] = -np.log(1 - (f.out3a / f.tot3a).clip(upper=0.6))
    return f.drop(columns=['out3a', 'tot3a']).add_suffix(sfx)

def gini(x):
    x = np.sort(x); n = len(x)
    return (2 * np.arange(1, n + 1) - n - 1) @ x / (n * x.sum()) if n == TOP_N and x.sum() > 0 else np.nan

act = ros[~ros.known_out].sort_values('min_pg', ascending=False).groupby(['gid', 'team_id']).head(TOP_N)
team = team_feats(ros, 'known_out').join(team_feats(ros, 'out', '_any')).join(act.groupby(['gid', 'team_id']).exp3a.apply(lambda s: gini(s.values)).rename('gini'))

# player-level: recent vs season-to-date 3PA rate, and catch-and-shoot share
h = app.sort_values(['player_id', 'game_date']).copy()
LG_RATE, B = logs.fg3a.sum() / logs['min'].sum(), 72.0          # league 3PA per minute; 72 pseudo-minutes of shrinkage
A = LG_RATE * B
for c in ['fg3a', 'min']:
    h[c + '_l5'] = h.groupby('player_id')[c].transform(lambda s: s.shift(1).rolling(5, min_periods=3).sum())
    h[c + '_szn'] = h.groupby(['player_id', 'season'])[c].transform(lambda s: s.shift(1).expanding().sum())
for c in ['cs3a', 'pu3a']:
    h[c + '_l20'] = h.groupby('player_id')[c].transform(lambda s: s.shift(1).rolling(20, min_periods=5).sum())
h['p36_dev'] = np.log((h.fg3a_l5 + A) / (h.min_l5 + B)) - np.log((h.fg3a_szn + A) / (h.min_szn + B))
h['cs_share'] = h.cs3a_l20 / (h.cs3a_l20 + h.pu3a_l20)

d = d.drop(columns=[c for c in d.columns if c in team.columns or c in ['p36_dev', 'cs_share', 'vac_crea_x_cs']])     # safe to rerun this cell
d = d.merge(team.reset_index().rename(columns={'gid': 'game_id'}), how='left', on=['game_id', 'team_id']).merge(h[['player_id', 'game_id', 'p36_dev', 'cs_share']], how='left', on=['player_id', 'game_id'])
d['vac_crea_x_cs'] = d.vac_crea * (d.cs_share - d.cs_share.median())
assert len(d) == len(y), 'the feature merge changed the row count: P0 / y / tr / te no longer line up'

FEATS = ['vac_usg', 'vac_crea', 'vac_3pa', 'gini', 'p36_dev', 'cs_share', 'vac_crea_x_cs']
print('coverage (share non-null):'); print(d[FEATS].notna().mean().round(3).to_string())
print('\nplayer-games with a known absence on the team: {:.1%}   with any absence (box score): {:.1%}'.format((d.n_out > 0).mean(), (d.n_out_any > 0).mean()))
d[FEATS + ['n_out', 'n_out_any']].describe().T.round(3)

In [ ]:
# Spot check: the team-games with the most vacated usage. Do the names and dates look right?
names = pd.read_sql('SELECT player_id, name FROM players', conn)
chk = ros[ros.known_out].merge(names, how='left', on='player_id').merge(tg[['game_id', 'team_id', 'game_date']].rename(columns={'game_id': 'gid'}), on=['gid', 'team_id'])
chk = chk[chk.gid.isin(d.game_id)]
chk.assign(vu=chk.min_pg / 48 * chk.usg_w).sort_values('vu', ascending=False)[['game_date', 'team_id', 'name', 'gp', 'min_pg', 'usg_w', 'exp3a', 'pa_pg', 'last_n', 'n']].head(15).round(2)

### Where does the zero miss sit?
The check from the design discussion: if the base is short on zeros mainly in games with an absence, the lineup terms can be the dispersion fix. If the gap is the same in normal-lineup games, dispersion needs its own terms regardless. This uses the box-score `out` flag, which is fine for a diagnosis.

In [ ]:
def zero_gap(mask, label):
    p, a, n = P0[mask, 0].mean(), (y[mask] == 0).mean(), mask.sum()
    return pd.Series({'n': n, 'P(0) pred': p, 'P(0) actual': a, 'gap': a - p, 'z': (a - p) * n / np.sqrt((P0[mask, 0] * (1 - P0[mask, 0])).sum())}, name=label)

any_out = (d.n_out_any > 0).values
pd.concat([zero_gap(~any_out, 'normal lineup'), zero_gap(any_out, 'rotation player out'), zero_gap((d.n_out > 0).values, 'known absence')], axis=1).T.round(3)

**Work through**
- Spot check: are the top rows real absences (injury, trade), or artifacts (a traded player counts as out for up to `ROT_WINDOW` games; the first games of a season have no window)?
- The z in the gap table treats player-games as independent, so read it as an upper bound.
- Look at the correlations: `d[FEATS].corr()`. `vac_usg`, `vac_crea` and `vac_3pa` will be highly correlated (the same players drive all three). Decide which to keep before fitting rather than letting the fit split one effect three ways.
- The window resets each season, so nobody is `known_out` in a team's first few games, and a player hurt since last season never becomes a candidate. Is that acceptable, or should the window reach back across the offseason for players still on the roster?
- A player out for 5 of the last 10 games is counted at his full per-game rate, but the teammates' rolling features have already half-adjusted. Would a decay on `last_n` be better?

## 6. Step B: lineup terms on top of dispersion
Features are standardized with the **training** mean and SD, so $\beta$ is the tilt per 1 SD and the prior means the same thing for each. Missing values become 0 (the training mean), i.e. "no information".

Three models, each compared with the base and with step A:
- `dispersion`: step A
- `lineup mean`: adds the mean tilt
- `lineup mean + spread`: also lets the spread depend on context (your hypothesis: a star out means more variation; a concentrated offense means role players swing more)

In [ ]:
MEAN_COLS = ['vac_usg', 'vac_crea', 'p36_dev']      # mean tilt (k)
DISP_COLS = ['vac_usg', 'gini']                     # spread (k^2)

mu, sg = d.loc[tr, FEATS].mean(), d.loc[tr, FEATS].std().replace(0, 1).fillna(1)    # a constant column would otherwise divide by zero
S = ((d[FEATS] - mu) / sg).fillna(0).add_prefix('z_')
ds = pd.concat([d, S], axis=1)

MODELS = {'dispersion': ([], []), 'lineup mean': (MEAN_COLS, []), 'lineup mean + spread': (MEAN_COLS, DISP_COLS)}
fits, preds, rows = {}, {}, []
affected = te & (d.n_out > 0).values
for name, (mc, dc) in MODELS.items():
    X, Z, sd = design(ds, ['z_' + c for c in mc], ['z_' + c for c in dc])
    fits[name] = (fit_layer(P0[tr], y[tr], X[tr], Z[tr], sd), X, Z, sd, mc, dc)
    preds[name] = predict_layer(fits[name][0], P0, X, Z)
    rows.append(compare(preds[name], P0, te, '{} vs base | all test'.format(name)))
    if name != 'dispersion':
        rows.append(compare(preds[name], preds['dispersion'], te, '{} vs dispersion | all test'.format(name)))
        if affected.sum() > 30: rows.append(compare(preds[name], preds['dispersion'], affected, '{} vs dispersion | known absence'.format(name)))
pd.concat(rows, axis=1).T.round(2)

In [ ]:
pd.concat([describe(P0[te], y[te], 'base')] + [describe(p[te], y[te], n) for n, p in preds.items()], axis=1).T.round(4)

**Work through**
- The row that matters is `... vs dispersion`. Negative and beyond ~2 SE means the lineup terms add something the dispersion fix did not.
- `known absence` rows: the layer moves predictions most there, so that is where a real effect should show. If it helps on all test rows but not on the affected ones, be suspicious.
- Swap features in `MEAN_COLS` / `DISP_COLS` one at a time (your one-change-at-a-time rule). Try `vac_3pa` in place of `vac_usg`, then add `vac_crea_x_cs`.
- Still to add from the design notes: the spread residual / morning-to-midday spread move (from `odds_featured.team_lines()`), and scenario-weighted PMFs for questionable players once the injury report is pulled.

## 7. Coefficient uncertainty and prior sensitivity
**Quadratic approximation** (Rethinking's `quap`): the posterior is approximated by a normal centered at the MAP, with covariance equal to the inverse Hessian of the negative log posterior. These SDs assume independent player-games, so they are too small where outcomes cluster; the held-out paired test above is the real evidence.

In [ ]:
def laplace(name):
    '''Posterior mode and SD for a fitted model, from a finite-difference Hessian of the gradient.'''
    th, X, Z, sd, mc, dc = fits[name]
    g = lambda t: neg_log_post(t, P0[tr], y[tr], X[tr], Z[tr], sd)[1]
    eps = 1e-5
    H = np.array([(g(th + eps * e) - g(th - eps * e)) / (2 * eps) for e in np.eye(len(th))])
    cov = np.linalg.inv((H + H.T) / 2)
    idx = ['beta0'] + ['beta: ' + c for c in mc] + ['gamma0'] + ['gamma: ' + c for c in dc] + ['delta']
    out = pd.DataFrame({'mode': th, 'sd': np.sqrt(np.diag(cov)), 'prior sd': sd}, index=idx)
    out['z'] = out['mode'] / out['sd']
    return out, cov

post, cov = laplace('lineup mean + spread')
post.round(4)

In [ ]:
# Prior sensitivity: scale every prior SD together and track the held-out log loss of the full model vs step A.
base_prior, sens = dict(PRIOR), []
for mult in [0.25, 0.5, 1, 2, 4, 100]:
    PRIOR = {k: v * mult for k, v in base_prior.items()}
    out = {}
    for name in ['dispersion', 'lineup mean + spread']:
        mc, dc = MODELS[name]
        X, Z, sd = design(ds, ['z_' + c for c in mc], ['z_' + c for c in dc])
        out[name] = predict_layer(fit_layer(P0[tr], y[tr], X[tr], Z[tr], sd), P0, X, Z)
    sens.append(compare(out['lineup mean + spread'], out['dispersion'], te, 'prior x{}'.format(mult)))
PRIOR = base_prior
pd.concat(sens, axis=1).T[['logloss diff x1000', 'se', 'z']].round(2)

**Work through**
- Which posterior SDs are close to their prior SD? Those coefficients learned nothing from the data: the prior is the answer.
- `prior x100` is effectively no prior (plain maximum likelihood). If held-out loss gets worse as the prior loosens, the regularization is earning its keep (Rethinking ch. 7).
- Is the sign of each $\beta$ what you predicted before looking? Write the prediction down first.
- Optional: draw from `np.random.default_rng().multivariate_normal(post['mode'], cov, 2000)` and push the draws through `predict_layer` for one player-game. The spread of P(over 2.5) across draws is the layer's own uncertainty, which is what the Bayesian pricing layer needs for sizing.

## 8. Calibration at the betting lines
The layer is only useful if it fixes P(over) where bets are placed. Predicted vs actual on the test dates, by line.

In [ ]:
BEST = 'lineup mean + spread'
rows = []
for line in [0.5, 1.5, 2.5, 3.5, 4.5]:
    r = {'line': line, 'actual over': (y[te] > line).mean(), 'base': P0[te][:, K > line].sum(axis=1).mean()}
    for name, p in preds.items(): r[name] = p[te][:, K > line].sum(axis=1).mean()
    rows.append(r)
pd.DataFrame(rows).set_index('line').round(4)

In [ ]:
# How far does the layer move individual prices? Change in P(over 2.5) on the test dates.
shift = preds[BEST][te][:, K > 2.5].sum(axis=1) - P0[te][:, K > 2.5].sum(axis=1)
plt.figure(figsize=(6, 3)); plt.hist(shift, bins=50); plt.xlabel('adjusted - base P(over 2.5)'); plt.show()
print('share of test player-games moved by 2+ pts: {:.1%}'.format((np.abs(shift) >= 0.02).mean()))

**Work through**
- Compare with the Nov 8 number: the base was ~4.5 pts too high on P(over 0.5).
- The stronger test, once this looks stable: does the adjusted PMF earn more weight than the base in the model/book blend? Books already price lineup news, so beating the base is necessary but not enough.

## 9. Save base and adjusted side by side
Parquet only, nothing goes to the database. Keep both so a later bet can be traced to the layer that priced it (backlog 0a).

In [ ]:
SAVE = False
if SAVE:
    out = d[['player_id', 'game_id', 'game_date', 'y'] + PC].copy()
    out[['adj_p{}'.format(k) for k in K]] = preds[BEST]
    out['is_test'] = te
    path = os.path.join(ROOT, 'nba', 'data', 'second_layer')
    os.makedirs(path, exist_ok=True)
    out.to_parquet(os.path.join(path, 'layer_{}.parquet'.format(d.game_date.max())), index=False)
    print('saved {:,} rows'.format(len(out)))